# Credit Risk Pipeline

An end-to-end credit risk scoring pipeline. This notebook:
1. Installs required packages
2. Downloads the credit risk dataset from Kaggle
3. Cleans the data
4. Trains an XGBoost model with cross-validation + probability calibration
5. Scores a sample applicant

**Before running:** get a Kaggle API token from https://www.kaggle.com/settings -> "Create New Token" (downloads `kaggle.json`)

## 1. Install dependencies

In [ ]:
!pip install -q kagglehub xgboost scikit-learn pandas numpy

## 2. Download the dataset from Kaggle

In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("laotse/credit-risk-dataset")
print("Path to dataset files:", path)

## 3. Imports

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

## 4. Load the CSV into a DataFrame

In [ ]:
csv_files = glob.glob(os.path.join(path, "*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in directory: {path}")

df = pd.read_csv(csv_files[0])
print(f"Dataset Loaded Successfully. Shape: {df.shape}")
df.head()

## 5. Clean data

In [ ]:
def clean_credit_data(df: pd.DataFrame) -> pd.DataFrame:
    df_clean = df.copy()

    if 'person_age' in df_clean.columns:
        df_clean = df_clean[df_clean['person_age'] < 100]
    if 'person_emp_length' in df_clean.columns:
        df_clean = df_clean[df_clean['person_emp_length'] < 60]

    print(f"Data cleaned. Remaining shape: {df_clean.shape}")
    return df_clean

df_clean = clean_credit_data(df)

## 6. Preprocessing pipeline builder

In [ ]:
def build_preprocessing_pipeline(num_cols: list, cat_cols: list) -> ColumnTransformer:
    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_pipeline, num_cols),
            ('cat', cat_pipeline, cat_cols)
        ]
    )
    return preprocessor

## 7. Train model with K-fold CV + calibration

In [ ]:
def train_credit_risk_model(df: pd.DataFrame, target_col: str, n_splits: int = 3):
    X = df.drop(columns=[target_col])
    y = df[target_col]

    num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    auc_scores = []

    print("\n--- Starting K-Fold Cross-Validation ---")
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        preprocessor = build_preprocessing_pipeline(num_cols, cat_cols)
        X_train_proc = preprocessor.fit_transform(X_train)
        X_val_proc = preprocessor.transform(X_val)

        ratio = (len(y_train) - sum(y_train)) / sum(y_train)

        model = XGBClassifier(
            n_estimators=150,
            learning_rate=0.08,
            max_depth=5,
            tree_method='hist',
            n_jobs=-1,
            scale_pos_weight=ratio,
            eval_metric='logloss',
            random_state=42
        )

        model.fit(X_train_proc, y_train)
        val_preds_prob = model.predict_proba(X_val_proc)[:, 1]

        auc = roc_auc_score(y_val, val_preds_prob)
        auc_scores.append(auc)
        print(f"Fold {fold} ROC-AUC: {auc:.4f}")

    print(f"Mean ROC-AUC across {n_splits} folds: {np.mean(auc_scores):.4f}\n")

    final_preprocessor = build_preprocessing_pipeline(num_cols, cat_cols)
    X_full_proc = final_preprocessor.fit_transform(X)

    base_model = XGBClassifier(
        n_estimators=150,
        learning_rate=0.08,
        max_depth=5,
        tree_method='hist',
        n_jobs=-1,
        scale_pos_weight=(len(y) - sum(y)) / sum(y),
        eval_metric='logloss',
        random_state=42
    )

    calibrated_model = CalibratedClassifierCV(estimator=base_model, method='sigmoid', cv=2, n_jobs=-1)
    calibrated_model.fit(X_full_proc, y)

    return calibrated_model, final_preprocessor, X.columns.tolist()

model, preprocessor, training_columns = train_credit_risk_model(
    df_clean, target_col='loan_status', n_splits=3
)

## 8. Helper: align new applicant data to training schema

In [ ]:
def align_to_training_schema(sample_df: pd.DataFrame, training_columns: list) -> pd.DataFrame:
    """
    Ensures a new (e.g. single-applicant) DataFrame has exactly the columns
    the preprocessor was fit on, in the same order. Missing columns are
    added as NaN (handled by the pipeline's imputers); extra columns are
    dropped.
    """
    missing = set(training_columns) - set(sample_df.columns)
    extra = set(sample_df.columns) - set(training_columns)

    if missing:
        print(f"Warning: sample is missing columns {missing}. Filling with NaN (will be imputed).")
    if extra:
        print(f"Warning: sample has unexpected extra columns {extra}. Dropping them.")

    return sample_df.reindex(columns=training_columns)

## 9. Score a sample applicant

In [ ]:
sample_applicant = {
    'person_age': 23,
    'person_income': 38000,
    'person_home_ownership': 'RENT',
    'person_emp_length': 1,
    'loan_intent': 'PERSONAL',
    'loan_grade': 'D',
    'loan_amnt': 28000,
    'loan_int_rate': 15.5,
    'loan_percent_income': round(28000 / 38000, 2),  # 0.74
    'cb_person_default_on_file': 'Y',
    'cb_person_cred_hist_length': 2
}

sample_df = pd.DataFrame([sample_applicant])
sample_df = align_to_training_schema(sample_df, training_columns)

sample_proc = preprocessor.transform(sample_df)
pd_estimate = model.predict_proba(sample_proc)[0, 1]

print(f"Calculated PD for Applicant: {pd_estimate:.2%}")